In [ ]:
from pathlib import Path
import os
import numpy as np
import arviz as az
import pickle
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import h5py  # Make sure h5py initialises properly before pandas ruins it
import warnings

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, plot_post_prior_comparison
from emu_renewal.utils import get_param_dim

set_matplotlib_formats("svg")

In [ ]:
param_display_names = {
    "admit_mean": "time to admission, mean",
    "admit_sd": "time to admission, SD",
    "cdr": "case detection proportion",
    "cross_immunity": "between-strain cross immunity",
    "death_mean": "time to death, mean",
    "death_sd": "time to death, SD",
    "gen_mean": "generation time, mean",
    "gen_sd": "generation time, SD",
    "har": "hospital admission proportion",
    "ifr": "infection fatality proportion",
    "relinfect": "variant relative infectiousness",
    "report_mean": "time to notification, mean",
    "report_sd": "time to notification, SD",
    "rt_init": "variable process initial value",
    "seed_offsets": "variant seeding offset",
    "seed_rates": "variant seeding rate",
    "shared_dispersion": "shared target dispersion",
    "stay_mean": "hospitalisation duration, mean",
    "stay_sd": "hospitalisation duration, SD",
}

In [ ]:
import pycountry
pycountry.countries.lookup("France")

In [ ]:
colours = {
    "no_mob": "black",
    "g_mob": "green",
    "fb_mob": "blue",
    "a_mob": "red",
}

def get_prior_vals_from_dist(x_vals, dist, d):
    multi_dist = len(dist.batch_shape) > 0
    logp = dist.log_prob
    log_vals = logp(x_vals[:, None])[:, d] if multi_dist else logp(x_vals)
    return np.exp(log_vals)

def plot_prior_multipost(idatas, n_cols, priors, var_names):
    idata = idatas["no_mob"]
    params = [p for p in idata.posterior if "proc" not in p]
    n_params = sum([get_param_dim(p, idata) for p in params])
    n_rows = int(np.ceil(n_params / n_cols))
    fig, ax = plt.subplots(n_rows, n_cols, figsize=[15, 14])
    axes = ax.ravel()
    n_ax = 0
    for p in params:
    
        # Posteriors
        for a in idatas:
            post = idatas[a].posterior[p]
            az.plot_density(post, ax=axes[n_ax:], hdi_prob=0.99, colors=[colours[a]])
    
        # Prior
        p_dim = get_param_dim(p, idata)
        for d in range(p_dim):
            # print(p)
            # print(d)
            axis = axes[n_ax]
            x_vals = np.linspace(*axis.get_xlim(), 100)
            y_vals = get_prior_vals_from_dist(x_vals, priors[p], d)
            axis.fill_between(x_vals, y_vals, color="k", alpha=0.2)
            n_ax += 1
            # display_name = param_display_names[p] if p in param_display_names else p
            display_name = get_display_name(p, p_dim, d, var_names)
            axis.set_title(display_name)
        
    for a in range(n_ax, len(axes)):
        axes[a].set_axis_off()

    fig.tight_layout()

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
c = "AUS"
analyses = {d.name: Path(d.path) for d in os.scandir(job_path / c) if d.is_dir()}

no_mob_path = analyses["no_mob"]
targets = load_targets(no_mob_path)
priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
country = pycountry.countries.lookup(c).name

In [ ]:
var_names = ["start"] + [v[5:] for v in targets.keys() if v.startswith("prop_")]

In [ ]:
get_param_dim

In [ ]:
var_map = {
    "start": "start strain",
    "alpha": "Alpha",
    "delta": "Delta",
    "ba2": "BA.2",
    "ba5": "BA.5",
}

def get_display_name(p, p_dim, d, var_names):
    d = d + 1 if p == "relinfect" else d
    var_ext = "" if p_dim == 1 else f", {var_map[var_names[d]]}"
    return param_display_names[p] + var_ext

In [ ]:
plot_prior_multipost(idatas, 4, priors, var_names)

In [ ]:
get_display_name("relinfect", 2, 0, ["ba2", "ba5"])